[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-06-pipeline-integration.ipynb#scrollTo=bb220001)

---
# Day 6 · Integrating Soda Scans into Airflow, Prefect, and CI/CD
**certified-journeys / sodacore-certified** · Soda Core for Data Quality · Practice Badge

> **Goal for today:** Use the Soda Core programmatic scan API to embed quality checks into Python pipelines. Learn the Airflow PythonOperator pattern, Prefect task pattern, GitHub Actions CI integration, and scan-gating logic that blocks downstream steps on failure.

In [ ]:
%pip install -q soda-core-duckdb

## Setup — Fixture Database

All examples in this notebook use a DuckDB fixture database with clean data. We will deliberately break it in specific cells to demonstrate FAIL states.

In [ ]:
import duckdb, tempfile, pathlib

tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "pipeline.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE orders AS SELECT * FROM (VALUES
        (1, 1, 150.00, 'pending',   '2024-01-10'),
        (2, 2, 320.50, 'shipped',   '2024-01-11'),
        (3, 3, 200.00, 'delivered', '2024-01-12'),
        (4, 1,  75.00, 'pending',   '2024-01-13')
    ) t(id, customer_id, amount, status, order_date)
""")
conn.close()

config_yml = f"""
data_sources:
  pipeline_db:
    type: duckdb
    path: "{db_path}"
"""

config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)
print("Fixture database ready:", db_path)

## Step 1 · The Programmatic Scan API

The **`soda.scan.Scan`** class is Soda Core's Python entrypoint. It replaces the `soda scan` CLI for embedding in code.

```python
from soda.scan import Scan

scan = Scan()
scan.set_data_source_name("my_datasource")   # must match key in configuration.yml
scan.add_configuration_yaml_file("path/to/configuration.yml")
scan.add_sodacl_yaml_file("path/to/checks.yml")
scan.execute()

exit_code = scan.get_exit_code()   # 0=pass, 1=warn, 2=fail
logs      = scan.get_logs_text()   # human-readable scan log
results   = scan.get_scan_results()  # structured dict for programmatic use
```

📖 https://docs.soda.io/soda-core/programmatic.html

In [ ]:
from soda.scan import Scan

checks_yml = """
checks for orders:
  - row_count > 0
  - missing_count(amount) = 0
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
"""

checks_path = tmpdir / "checks.yml"
checks_path.write_text(checks_yml)

def run_soda_scan(data_source: str, config_path: str, checks_path: str) -> int:
    """Run a Soda scan and return the exit code."""
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()
    print(scan.get_logs_text())
    return scan.get_exit_code()

exit_code = run_soda_scan(
    data_source="pipeline_db",
    config_path=str(config_path),
    checks_path=str(checks_path),
)

print(f"Exit code: {exit_code}")

### What just happened?

- **`run_soda_scan()`** is the reusable wrapper you'll drop into any Python pipeline.
- It returns an **exit code**: `0` = all pass, `1` = warnings, `2` = at least one failure.
- Keeping the logic in a function makes it trivial to call from Airflow, Prefect, or a shell script.
- The function prints logs to stdout — in production, redirect these to your logging framework.

## Step 2 · Scan-Gating — Blocking Downstream Steps

A **scan gate** raises an exception if any check fails, preventing downstream pipeline steps from running on bad data.

```python
def run_and_gate(data_source, config_path, checks_path, strict=False):
    exit_code = run_soda_scan(data_source, config_path, checks_path)
    if exit_code == 2:
        raise ValueError("Data quality FAIL — downstream step blocked")
    if strict and exit_code == 1:
        raise ValueError("Data quality WARN treated as FAIL in strict mode")
    return exit_code
```

The `strict` flag lets you promote warnings to failures in production pipelines while keeping them as warnings during development.

In [ ]:
def run_and_gate(data_source: str, config_path: str, checks_path: str,
                 strict: bool = False) -> int:
    """Run a Soda scan and raise on failure (or warning in strict mode)."""
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()

    exit_code = scan.get_exit_code()
    print(scan.get_logs_text())

    if exit_code == 2:
        raise ValueError("[GATE] Data quality FAIL — downstream step blocked.")
    if strict and exit_code == 1:
        raise ValueError("[GATE] Data quality WARN treated as FAIL (strict=True).")
    return exit_code

# ── Test with clean data (should pass) ────────────────────────────────────
print("=== Clean data ===")
try:
    code = run_and_gate("pipeline_db", str(config_path), str(checks_path))
    print(f"Gate passed. Exit code: {code}. Downstream steps can run.")
except ValueError as e:
    print(f"Gate blocked: {e}")

In [ ]:
# ── Inject a bad row, then re-run to demonstrate the gate blocking ─────────
conn2 = duckdb.connect(db_path)
conn2.execute("INSERT INTO orders VALUES (5, 4, NULL, 'unknown', '2024-01-14')")
conn2.close()

print("=== Data with failures injected ===")
try:
    code = run_and_gate("pipeline_db", str(config_path), str(checks_path))
    print(f"Gate passed. Downstream steps can run.")
except ValueError as e:
    print(f"\n{e}")
    print("Downstream ETL step would NOT run.")

### What just happened?

- First scan (clean data): exit code 0 → gate **passes** → downstream step would execute.
- After injecting `amount=NULL` and `status='unknown'`: exit code 2 → gate **raises** → downstream step is blocked.
- The `ValueError` propagates through the call stack — in Airflow this marks the task FAILED; in Prefect it marks the flow run FAILED.
- The `strict` flag is useful for pre-production environments where you want zero tolerance for warnings.

## Step 3 · Apache Airflow Integration Pattern

In an Airflow DAG, wrap `run_and_gate` in a `PythonOperator`. If the scan fails, the task fails and Airflow marks all downstream tasks as skipped or failed (depending on trigger rules).

```python
# airflow_dag_example.py  (not executed here — requires airflow installed)
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def soda_quality_check():
    """Airflow task: run Soda scan and fail the task on DQ failure."""
    from soda.scan import Scan
    scan = Scan()
    scan.set_data_source_name("my_warehouse")
    scan.add_configuration_yaml_file("/opt/airflow/soda/configuration.yml")
    scan.add_sodacl_yaml_file("/opt/airflow/soda/checks/orders.yml")
    scan.execute()
    if scan.get_exit_code() == 2:
        from airflow.exceptions import AirflowException
        raise AirflowException("Soda quality check FAILED — see scan logs")

with DAG("etl_pipeline", start_date=datetime(2024, 1, 1), schedule="@daily") as dag:
    extract   = PythonOperator(task_id="extract",   python_callable=extract_fn)
    dq_check  = PythonOperator(task_id="dq_check",  python_callable=soda_quality_check)
    transform = PythonOperator(task_id="transform", python_callable=transform_fn)

    extract >> dq_check >> transform
    # transform only runs if dq_check passes
```

📖 https://docs.soda.io/soda/orchestrate-scans.html

In [ ]:
# ── Simulate the Airflow task function (no airflow import needed) ──────────
class AirflowException(Exception):
    """Simulated Airflow exception for demonstration."""
    pass

def soda_quality_check_airflow(data_source, config_path, checks_path):
    """Airflow-style task: raise AirflowException on DQ failure."""
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()
    print(scan.get_logs_text())
    if scan.get_exit_code() == 2:
        raise AirflowException("Soda quality check FAILED — downstream tasks will not run")
    return scan.get_exit_code()

# With bad data still in the DB
print("Simulating Airflow dq_check task:")
try:
    soda_quality_check_airflow("pipeline_db", str(config_path), str(checks_path))
except AirflowException as e:
    print(f"AirflowException raised: {e}")
    print("Airflow marks task FAILED; transform task is NOT triggered.")

### What just happened?

- The function signature matches what Airflow's `PythonOperator(python_callable=...)` expects.
- Raising `AirflowException` inside a PythonOperator marks the Airflow task as **FAILED** and all downstream tasks as **SKIPPED** (or UPSTREAM_FAILED, depending on trigger rules).
- Keep the imports inside the function body — Airflow workers execute tasks in a fresh Python process where `airflow` is always available but other packages may need careful dependency management.

## Step 4 · Prefect Integration Pattern

In Prefect, decorate the scan function with `@task`. If the task raises an exception, Prefect marks the flow run FAILED.

```python
# prefect_flow_example.py  (not executed here — requires prefect installed)
from prefect import flow, task

@task(name="soda-quality-check")
def soda_quality_check(data_source: str, config_path: str, checks_path: str) -> dict:
    from soda.scan import Scan
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()
    results = scan.get_scan_results()
    if scan.get_exit_code() == 2:
        raise ValueError("Soda quality check FAILED")
    return results

@flow(name="etl-pipeline")
def etl_pipeline():
    raw_data = extract_task()          # upstream task
    dq_results = soda_quality_check(   # gate
        data_source="my_dw",
        config_path="/soda/configuration.yml",
        checks_path="/soda/checks/orders.yml",
    )
    transform_task(dq_results)         # only runs if gate passes
```

📖 https://docs.soda.io/soda/orchestrate-scans.html

In [ ]:
# ── Simulate the Prefect @task function (no prefect import needed) ─────────
def soda_quality_check_prefect(data_source: str, config_path: str,
                                checks_path: str) -> dict:
    """Prefect-style task: return results dict or raise on failure."""
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()

    results = scan.get_scan_results()
    exit_code = scan.get_exit_code()

    # Attach summary to results for Prefect artifact
    results["exit_code"] = exit_code
    results["status"] = ["pass", "warn", "fail"][min(exit_code, 2)]

    if exit_code == 2:
        raise ValueError(f"Soda DQ FAILED. See scan results: {results.get('status')}")
    return results

print("Simulating Prefect @task soda_quality_check:")
try:
    res = soda_quality_check_prefect("pipeline_db", str(config_path), str(checks_path))
    print(f"Task succeeded. Status: {res.get('status')}")
except ValueError as e:
    print(f"Task raised: {e}")
    print("Prefect marks flow run FAILED; downstream tasks not executed.")

### What just happened?

- The Prefect pattern returns a `results` dict on success — downstream tasks can consume these results as inputs.
- Returning the scan results as a task output allows Prefect to store them as an **artifact** in the UI.
- The `status` key ('pass', 'warn', 'fail') gives downstream tasks a simple way to conditionally branch.
- In production, add `@task(retries=2, retry_delay_seconds=60)` to handle transient DB connection issues.

## Step 5 · GitHub Actions CI Pattern

Add a Soda scan step to CI so every pull request is validated against a DuckDB fixture database before merge.

```yaml
# .github/workflows/data-quality.yml
name: Data Quality Check
on: [pull_request]

jobs:
  soda-scan:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install Soda Core
        run: pip install soda-core-duckdb

      - name: Create fixture database
        run: python tests/fixtures/create_fixture_db.py

      - name: Run Soda scan
        run: python ci/run_soda.py
        # ci/run_soda.py: calls run_and_gate() and sys.exit(scan.get_exit_code())
```

The key is `sys.exit(scan.get_exit_code())` — a non-zero exit code marks the CI step as **failed** and blocks PR merge.

In [ ]:
# ── Simulate the ci/run_soda.py script ────────────────────────────────────
import sys

def ci_soda_entrypoint(data_source: str, config_path: str,
                        checks_path: str) -> int:
    """
    CI entrypoint: run scan and return exit code.
    In a real CI script: sys.exit(ci_soda_entrypoint(...))
    """
    scan = Scan()
    scan.set_data_source_name(data_source)
    scan.add_configuration_yaml_file(config_path)
    scan.add_sodacl_yaml_file(checks_path)
    scan.execute()
    print(scan.get_logs_text())
    exit_code = scan.get_exit_code()
    status_emoji = {0: "✅", 1: "⚠️", 2: "❌"}.get(exit_code, "?")
    print(f"\n{status_emoji} CI scan exit code: {exit_code}")
    if exit_code == 2:
        print("CI step FAILED — PR merge blocked")
    elif exit_code == 1:
        print("CI step WARN — PR can merge but quality review recommended")
    else:
        print("CI step PASS — PR ready to merge")
    return exit_code

# Simulate CI run with bad data
print("=== Simulating GitHub Actions ci step ===")
ci_exit = ci_soda_entrypoint("pipeline_db", str(config_path), str(checks_path))
# In a real script: sys.exit(ci_exit)

### What just happened?

- **`sys.exit(exit_code)`** at the end of a CI script propagates the exit code to the shell → GitHub Actions treats any non-zero exit as a failed step.
- The fixture database is created by a setup script before the scan runs — this makes the scan deterministic and independent of the production database.
- Add this CI check as a **required status check** on your main branch to enforce data quality for every schema migration.
- Store `configuration.yml` without credentials; use **GitHub Secrets** to inject credentials at runtime via `${DB_PASSWORD}`.

## Step 6 · Per-Check Result Inspection

For fine-grained gating — for example, blocking only on **critical** check failures while allowing non-critical ones to warn — inspect individual check results from `get_scan_results()`.

In [ ]:
# Run scan and inspect per-check outcomes
scan_final = Scan()
scan_final.set_data_source_name("pipeline_db")
scan_final.add_configuration_yaml_file(str(config_path))
scan_final.add_sodacl_yaml_file(str(checks_path))
scan_final.execute()

results = scan_final.get_scan_results()

# The scan results dict contains check-level outcomes
print("Top-level keys:", list(results.keys()) if isinstance(results, dict) else type(results))
print()
print("Exit code:", scan_final.get_exit_code())
print()

# Summary pattern: count pass/warn/fail
logs = scan_final.get_logs_text()
pass_count = logs.count("check result: \x1b[92mPASS\x1b[0m") + logs.count("PASS")
fail_count = logs.count("FAIL")
warn_count = logs.count("WARN")
print(f"Scan summary — checks with FAIL: {fail_count}, WARN: {warn_count}")

### What just happened?

- `get_scan_results()` returns the full structured dict; the exact schema depends on the Soda Core version.
- For simple gating, `get_exit_code()` is almost always sufficient.
- For selective gating (e.g., block on PK failures but not on volume warnings), parse `get_logs_text()` or extend the checks YAML to add labels and filter results by label in code.
- In production, send the `get_scan_results()` dict to your observability platform (Datadog, Grafana) as a structured log event.

## Step 7 · Selective Gating with Multiple Checks Files

Separate checks into **critical** and **non-critical** files. Gate on the critical scan; log warnings from the non-critical scan.

In [ ]:
critical_checks = """
# critical_checks.yml — FAIL here blocks the pipeline
checks for orders:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
"""

advisory_checks = """
# advisory_checks.yml — WARN here is logged but does not block
checks for orders:
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
  - missing_count(amount):
      warn: when > 0
      fail: when > 5
"""

critical_path  = tmpdir / "critical_checks.yml"
advisory_path  = tmpdir / "advisory_checks.yml"
critical_path.write_text(critical_checks)
advisory_path.write_text(advisory_checks)

def run_selective_gate(data_source, config_path, critical_path, advisory_path):
    # 1. Critical scan — gate the pipeline
    scan_c = Scan()
    scan_c.set_data_source_name(data_source)
    scan_c.add_configuration_yaml_file(str(config_path))
    scan_c.add_sodacl_yaml_file(str(critical_path))
    scan_c.execute()
    print("── Critical scan ──")
    print(scan_c.get_logs_text())
    if scan_c.get_exit_code() == 2:
        raise ValueError("Critical DQ check FAILED — pipeline halted")

    # 2. Advisory scan — log but do not block
    scan_a = Scan()
    scan_a.set_data_source_name(data_source)
    scan_a.add_configuration_yaml_file(str(config_path))
    scan_a.add_sodacl_yaml_file(str(advisory_path))
    scan_a.execute()
    print("── Advisory scan ──")
    print(scan_a.get_logs_text())
    print(f"Advisory exit code: {scan_a.get_exit_code()} (not blocking)")

try:
    run_selective_gate("pipeline_db", config_path, critical_path, advisory_path)
    print("\nPipeline continues past gate.")
except ValueError as e:
    print(f"\nGate blocked: {e}")

### What just happened?

- **Critical checks** (PK, row count) pass → pipeline proceeds past the gate.
- **Advisory checks** (status validity, missing amount) fail → logged but do NOT block the pipeline.
- This two-tier pattern is the recommended production approach: critical checks gate the pipeline, advisory checks feed a quality dashboard.
- Rotate checks from advisory to critical as confidence in the upstream source grows.

## Challenge

```python
# Challenge: Build a full scan runner with logging and notification.
#
# 1. Write a function run_with_summary(data_source, config_path, checks_path)
#    that:
#      a. Runs the scan
#      b. Parses get_logs_text() to count PASS / WARN / FAIL checks
#      c. Prints a formatted summary table:
#            ┌──────────────────┬────────┐
#            │ Check status     │ Count  │
#            ├──────────────────┼────────┤
#            │ PASS             │ 2      │
#            │ WARN             │ 0      │
#            │ FAIL             │ 1      │
#            └──────────────────┴────────┘
#      d. Returns the exit code
#
# 2. Call it on the pipeline_db with the critical_checks.yml
# 3. Add a 'notify' parameter: if notify=True and exit_code==2,
#    print "ALERT: Would send notification to team channel"
#    (use a placeholder — never hardcode a real webhook URL)

# Your solution here
```

---
## Day 6 key concepts recap

| Pattern | Key API / method | Use case |
|---|---|---|
| Programmatic scan | `Scan()`, `scan.execute()` | Any Python pipeline |
| Exit code gate | `scan.get_exit_code() == 2` | Block on failure |
| Airflow integration | `PythonOperator` + `AirflowException` | DAG-based pipelines |
| Prefect integration | `@task` + `raise ValueError` | Flow-based pipelines |
| GitHub Actions CI | `sys.exit(exit_code)` in CI script | PR merge gate |
| Selective gating | Critical vs advisory checks files | Tiered quality enforcement |
| Result inspection | `scan.get_scan_results()` | Structured logging, alerting |

> **Tip:** When embedding Soda in Airflow, use `scan.get_scan_results()` to inspect individual check outcomes and only raise exceptions for checks tagged as critical — this lets non-critical quality warnings surface without stopping the DAG.

---
## What's next
**Day 7** → Soda Cloud — Publishing Scan Results, Alerting, and Data Contracts. You'll connect Soda Core to Soda Cloud, publish scan results to the health dashboard, configure FAIL alerts, and write your first data contract.

Mark Day 6 complete in your [tracker](../index.html).